# 🧹 Limpeza e Padronização — Camada Silver (SISCAN / 10ª CRS)

Este notebook transforma os dados brutos da **Camada Bronze** em dados limpos, padronizados e estruturados na **Camada Silver** da Arquitetura Medallion.

---

## Limpezas aplicadas:

1. **Separação código/nome do município** — coluna `Munic.de residencia` (ex: `430040 ALEGRETE`) é dividida em `codigo_municipio` (int) e `municipio` (string)
2. **Remoção de linhas de totalização** — linhas `Total` agregadas pelo TABNET são removidas (mantém-se apenas dados por município)
3. **Correção de encoding** — nomes de colunas com caracteres corrompidos (ex: `MAR�O` → `MARCO`) são normalizados
4. **Cast de tipos** — colunas numéricas lidas como STRING são convertidas para INTEGER
5. **Unpivot (wide → long)** — formato matricial (meses/faixas etárias como colunas) é convertido para formato tidy (uma linha por observação)
6. **Extração de ano e categoria do nome do arquivo** — quando os dados vêm de múltiplos CSVs por ano ou por categoria, o metadado é extraído de `_file_source`
7. **Padronização de nomes de colunas** — snake_case, sem espaços nem acentos
8. **Tratamento de nulos** — valores ausentes em colunas numéricas são substituídos por 0
9. **Rastreabilidade** — colunas `_bronze_source` e `_silver_processed_time` preservam a origem e o momento do processamento

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
import re

# Catálogo e schema
CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

# 11 códigos IBGE oficiais da 10ª CRS / RS
CODIGOS_10CRS = [430040, 430187, 431060, 431171, 431175, 431530, 431640, 431697, 431710, 431830, 432240]

# Faixas etárias que compõem a população-alvo (25-64 anos) — diretrizes MS rastreamento colo do útero
FAIXAS_ALVO = ["25 a 29 anos", "30 a 34 anos", "35 a 39 anos", "40 a 44 anos", "45 a 49 anos", "50 a 54 anos", "55 a 59 anos", "60 a 64 anos"]

# Criar schema silver se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")

print(f"Schema {CATALOG}.{SCHEMA_SILVER} garantido.")
print(f"Filtro 10ª CRS: {len(CODIGOS_10CRS)} municípios")

## 1. Limpeza: `exames_geral_raw` → `silver.exames_geral`

**Origem:** 1 arquivo CSV com todos os anos (2020-2025), formato wide (72 meses como colunas).

**Transformações:**
* Separar `codigo_municipio` e `municipio` da coluna `Munic.de residencia`
* Remover linha `Total`
* Unpivot dos 72 meses para formato long (`ano`, `mes`, `quantidade`)
* Remover coluna ` Total` agregada
* Padronizar nomes dos meses

In [0]:
# ---------------------------------------------------------------------------
# 1. SILVER: exames_geral (Q1 + Q2) — Unpivot meses para formato long
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.exames_geral_raw")

# 1a. Separar código e nome do município (initcap + remove barras invertidas)
df_silver = (
    df_bronze
    .withColumn("codigo_municipio", F.expr("try_cast(substring(`Munic.de residencia`, 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
)

# 1b. Filtro estrito: apenas municípios da 10ª CRS
df_silver = df_silver.filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))

# 1c. Identificar colunas de meses (formato MES/ANO) dinamicamente
all_cols = df_bronze.columns
month_cols = [c for c in all_cols if "/" in c and c != "Munic.de residencia"]

# 1d. Unpivot: wide → long
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio"],
    values=month_cols,
    variableColumnName="mes_ano_col",
    valueColumnName="quantidade"
)

# 1e. Extrair ano e mes a partir do nome da coluna (ex: 'JANEIRO/2020')
# Mapeamento de nomes de meses (incluindo encoding corrompido MAR�O)
month_map = {
    "JANEIRO": 1, "FEVEREIRO": 2, "MARCO": 3, "MAR�O": 3, "MARÇO": 3,
    "ABRIL": 4, "MAIO": 5, "JUNHO": 6, "JULHO": 7, "AGOSTO": 8,
    "SETEMBRO": 9, "OUTUBRO": 10, "NOVEMBRO": 11, "DEZEMBRO": 12
}

month_expr = "CASE "
for name, num in month_map.items():
    month_expr += f"WHEN UPPER(SUBSTR(mes_ano_col, 1, INSTR(mes_ano_col, '/') - 1)) = '{name}' THEN {num} "
month_expr += "ELSE NULL END"

df_long = (
    df_long
    .withColumn("ano", F.expr("try_cast(regexp_extract(mes_ano_col, '/([0-9]{4})$', 1) as int)"))
    .withColumn("mes", F.expr(f"try_cast({month_expr} as int)"))
    .withColumn("quantidade", F.expr("try_cast(quantidade as int)"))
    .withColumn("dt_competencia", F.to_date(F.concat_ws("-", F.col("ano"), F.lpad(F.col("mes"), 2, "0"), F.lit("01")), "yyyy-MM-dd"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.exames_geral_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
)

# 1f. Selecionar e ordenar colunas finais
df_long = df_long.select(
    "codigo_municipio", "municipio", "ano", "mes", "dt_competencia", "quantidade",
    "_bronze_source", "_silver_processed_time"
).orderBy("codigo_municipio", "ano", "mes")

# 1g. Escrever na camada Silver
(
    df_long.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral")
)

print(f"exames_geral: {df_long.count()} linhas gravadas na Silver.")
display(df_long.limit(10))

## 2. Limpeza: `faixa_etaria_raw` → `silver.faixa_etaria`

**Origem:** 6 arquivos CSV (1 por ano 2020-2025), formato wide (faixas etárias como colunas, todas como STRING).

**Transformações:**
* Extrair `ano` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linha `Total`
* Cast de STRING → INTEGER em todas as colunas de faixa etária
* Unpivot das faixas etárias para formato long (`faixa_etaria`, `quantidade`)
* Nulos substituídos por 0

In [0]:
# ---------------------------------------------------------------------------
# 2. SILVER: faixa_etaria (Q3) — Extrair ano do arquivo + unpivot faixas
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.faixa_etaria_raw")

# 2a. Extrair ano do nome do arquivo (ex: ..._10CRS_2020.csv -> 2020)
df_silver = df_bronze.withColumn("ano", F.expr("try_cast(regexp_extract(_file_source, '([0-9]{4})\\.csv$', 1) as int)"))

# 2b. Separar código e nome do município (initcap + remove barras invertidas)
df_silver = (
    df_silver
    .withColumn("codigo_municipio", F.expr("try_cast(substring(`Munic.de residencia`, 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
)

# 2c. Filtro estrito: apenas municípios da 10ª CRS
df_silver = df_silver.filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))

# 2d. Identificar colunas de faixa etária dinamicamente (excluir metadados e Total)
exclude_cols = {"Munic.de residencia", "_ingestion_time", "_file_source", " Total", "ano", "codigo_municipio", "municipio"}
faixa_cols = [c for c in df_silver.columns if c not in exclude_cols]

# 2e. Cast de STRING → INT e substituir nulos por 0 (com withColumns)
cast_exprs = {c: F.expr(f"coalesce(try_cast(`{c}` as int), 0)") for c in faixa_cols}
df_silver = df_silver.withColumns(cast_exprs)

# 2f. Unpivot: wide → long
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "ano"],
    values=faixa_cols,
    variableColumnName="faixa_etaria_raw",
    valueColumnName="quantidade"
)

# 2g. Normalizar nomes das faixas etárias para formato legível + flag populacao-alvo
df_long = (
    df_long
    .withColumn("faixa_etaria", F.trim(F.col("faixa_etaria_raw")))  # remover espaços à esquerda
    .withColumn("faixa_etaria", F.regexp_replace(F.col("faixa_etaria"), r"^Entre\s+", ""))  # remover prefixo "Entre "
    .withColumn("fl_populacao_alvo", F.when(F.col("faixa_etaria").isin(FAIXAS_ALVO), F.lit(True)).otherwise(F.lit(False)))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.faixa_etaria_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
)

# 2h. Selecionar colunas finais
df_long = df_long.select(
    "codigo_municipio", "municipio", "ano", "faixa_etaria", "fl_populacao_alvo", "quantidade",
    "_bronze_source", "_silver_processed_time"
).orderBy("codigo_municipio", "ano", "faixa_etaria")

# 2i. Escrever na camada Silver
(
    df_long.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.faixa_etaria")
)

print(f"faixa_etaria: {df_long.count()} linhas gravadas na Silver.")
display(df_long.limit(10))

## 3. Limpeza: `adequabilidade_raw` → `silver.adequabilidade`

**Origem:** 3 arquivos CSV (1 por categoria: Satisfatória, Insatisfatória, Rejeitada), formato wide (72 meses como colunas, tipos mistos STRING/INT).

**Transformações:**
* Extrair `categoria_adequabilidade` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linhas `Total`
* Cast de STRING → INTEGER em todas as colunas de meses (tipos mistos)
* Unpivot dos 72 meses para formato long (`ano`, `mes`, `quantidade`)
* Nulos substituídos por 0

In [0]:
# ---------------------------------------------------------------------------
# 3. SILVER: adequabilidade (Q4) — Extrair categoria do arquivo + unpivot meses
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.adequabilidade_raw")

# 3a. Extrair categoria de adequabilidade do nome do arquivo
# Ex: 'satisfatório%20-cito...' -> 'Satisfatória'; 'rejeitada%20-cito...' -> 'Rejeitada'
df_silver = (
    df_bronze
    .withColumn("categoria_adequabilidade",
        F.when(F.lower(F.col("_file_source")).contains("insatisfat"), F.lit("Insatisfatoria"))
         .when(F.lower(F.col("_file_source")).contains("satisfat"), F.lit("Satisfatoria"))
         .when(F.lower(F.col("_file_source")).contains("rejeitada"), F.lit("Rejeitada"))
         .otherwise(F.lit("Desconhecida"))
    )
)

# 3b. Separar código e nome do município (initcap + remove barras invertidas)
df_silver = (
    df_silver
    .withColumn("codigo_municipio", F.expr("try_cast(substring(`Munic.de residencia`, 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
)

# 3c. Filtro estrito: apenas municípios da 10ª CRS
df_silver = df_silver.filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))

# 3d. Identificar colunas de meses dinamicamente
all_cols = df_bronze.columns
month_cols = [c for c in all_cols if "/" in c and c != "Munic.de residencia"]

# 3e. Cast de todas as colunas de meses para INT (com withColumns)
cast_exprs = {c: F.expr(f"coalesce(try_cast(`{c}` as int), 0)") for c in month_cols}
df_silver = df_silver.withColumns(cast_exprs)

# 3f. Unpivot: wide → long
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "categoria_adequabilidade"],
    values=month_cols,
    variableColumnName="mes_ano_col",
    valueColumnName="quantidade"
)

# 3g. Extrair ano e mes (mesmo mapeamento da tabela exames_geral)
month_map = {
    "JANEIRO": 1, "FEVEREIRO": 2, "MARCO": 3, "MAR�O": 3, "MARÇO": 3,
    "ABRIL": 4, "MAIO": 5, "JUNHO": 6, "JULHO": 7, "AGOSTO": 8,
    "SETEMBRO": 9, "OUTUBRO": 10, "NOVEMBRO": 11, "DEZEMBRO": 12
}

month_expr = "CASE "
for name, num in month_map.items():
    month_expr += f"WHEN UPPER(SUBSTR(mes_ano_col, 1, INSTR(mes_ano_col, '/') - 1)) = '{name}' THEN {num} "
month_expr += "ELSE NULL END"

df_long = (
    df_long
    .withColumn("ano", F.expr("try_cast(regexp_extract(mes_ano_col, '/([0-9]{4})$', 1) as int)"))
    .withColumn("mes", F.expr(f"try_cast({month_expr} as int)"))
    .withColumn("quantidade", F.expr("try_cast(quantidade as int)"))
    .withColumn("dt_competencia", F.to_date(F.concat_ws("-", F.col("ano"), F.lpad(F.col("mes"), 2, "0"), F.lit("01")), "yyyy-MM-dd"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.adequabilidade_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
)

# 3h. Selecionar colunas finais
df_long = df_long.select(
    "codigo_municipio", "municipio", "categoria_adequabilidade",
    "ano", "mes", "dt_competencia", "quantidade",
    "_bronze_source", "_silver_processed_time"
).orderBy("codigo_municipio", "categoria_adequabilidade", "ano", "mes")

# 3i. Escrever na camada Silver
(
    df_long.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.adequabilidade")
)

print(f"adequabilidade: {df_long.count()} linhas gravadas na Silver.")
display(df_long.limit(10))

## 4. Limpeza: `resultados_raw` → `silver.resultados_citopatologicos`

**Origem:** 6 arquivos CSV (1 por ano 2020-2025), formato wide (faixas etárias como colunas STRING).

**Transformações:**
* Extrair `ano` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linhas `Total`
* Cast de STRING → INTEGER em todas as colunas de faixa etária
* Unpivot das faixas etárias para formato long (`faixa_etaria`, `quantidade`)
* Nulos substituídos por 0

In [0]:
# ---------------------------------------------------------------------------
# 4. SILVER: resultados_citopatologicos (Q5) — Extrair ano + unpivot faixas
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.resultados_raw")

# 4a. Extrair ano do nome do arquivo (ex: ...faixa_etaria_2020.csv -> 2020)
df_silver = df_bronze.withColumn("ano", F.expr("try_cast(regexp_extract(_file_source, '([0-9]{4})\\.csv$', 1) as int)"))

# 4b. Separar código e nome do município (initcap + remove barras invertidas)
df_silver = (
    df_silver
    .withColumn("codigo_municipio", F.expr("try_cast(substring(`Munic.de residencia`, 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
)

# 4c. Filtro estrito: apenas municípios da 10ª CRS
df_silver = df_silver.filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))

# 4d. Identificar colunas de faixa etária dinamicamente
exclude_cols = {"Munic.de residencia", "_ingestion_time", "_file_source", " Total", "ano", "codigo_municipio", "municipio"}
faixa_cols = [c for c in df_silver.columns if c not in exclude_cols]

# 4e. Cast de STRING → INT e substituir nulos por 0 (com withColumns)
cast_exprs = {c: F.expr(f"coalesce(try_cast(`{c}` as int), 0)") for c in faixa_cols}
df_silver = df_silver.withColumns(cast_exprs)

# 4f. Unpivot: wide → long
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "ano"],
    values=faixa_cols,
    variableColumnName="faixa_etaria_raw",
    valueColumnName="quantidade"
)

# 4g. Normalizar nomes das faixas etárias para formato legível + flag populacao-alvo
df_long = (
    df_long
    .withColumn("faixa_etaria", F.trim(F.col("faixa_etaria_raw")))
    .withColumn("faixa_etaria", F.regexp_replace(F.col("faixa_etaria"), r"^Entre\s+", ""))
    .withColumn("fl_populacao_alvo", F.when(F.col("faixa_etaria").isin(FAIXAS_ALVO), F.lit(True)).otherwise(F.lit(False)))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.resultados_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
)

# 4h. Selecionar colunas finais
df_long = df_long.select(
    "codigo_municipio", "municipio", "ano", "faixa_etaria", "fl_populacao_alvo", "quantidade",
    "_bronze_source", "_silver_processed_time"
).orderBy("codigo_municipio", "ano", "faixa_etaria")

# 4i. Escrever na camada Silver
(
    df_long.write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.resultados_citopatologicos")
)

print(f"resultados_citopatologicos: {df_long.count()} linhas gravadas na Silver.")
display(df_long.limit(10))

## 5. Validação da Camada Silver

Verificação de contagem de linhas, esquemas e amostragem das tabelas Silver criadas.

In [0]:
# ---------------------------------------------------------------------------
# 5. VALIDAÇÃO: Contagem de linhas, schemas e verificação de nulos/vazios
# ---------------------------------------------------------------------------

from pyspark.sql.types import DoubleType, FloatType

tables_silver = [
    ("exames_geral", "codigo_municipio, municipio, ano, mes, dt_competencia, quantidade"),
    ("faixa_etaria", "codigo_municipio, municipio, ano, faixa_etaria, fl_populacao_alvo, quantidade"),
    ("adequabilidade", "codigo_municipio, municipio, categoria_adequabilidade, ano, mes, dt_competencia, quantidade"),
    ("resultados_citopatologicos", "codigo_municipio, municipio, ano, faixa_etaria, fl_populacao_alvo, quantidade"),
]

print("=" * 70)
print(f"{'TABELA SILVER':<35} {'LINHAS':>10} {'COLUNAS':>10}")
print("=" * 70)

for table_name, _ in tables_silver:
    df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{table_name}")
    col_list = df.columns
    print(f"{table_name:<35} {df.count():>10} {len(col_list):>10}")

print("=" * 70)
print("\n✅ Camada Silver criada com sucesso!")

# ---------------------------------------------------------------------------
# 5.1 VALIDAÇÃO DE NULOS / VAZIOS EM TODAS AS TABELAS SILVER
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print(f"{'TABELA SILVER':<28} {'COLUNA':<30} {'NULOS/VAZIOS':>12}")
print("=" * 70)

for table_name, _ in tables_silver:
    df = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.{table_name}")
    col_list = df.columns
    schema = df.schema

    # Monta a contagem de nulos/vazios para cada coluna (isnan só para float/double)
    null_exprs = []
    for c in col_list:
        if isinstance(schema[c].dataType, (DoubleType, FloatType)):
            null_exprs.append(
                F.count(F.when(
                    F.col(c).isNull() |
                    (F.trim(F.col(c).cast("string")) == "") |
                    F.isnan(F.col(c)), c
                )).alias(c)
            )
        else:
            null_exprs.append(
                F.count(F.when(
                    F.col(c).isNull() |
                    (F.trim(F.col(c).cast("string")) == ""), c
                )).alias(c)
            )

    null_counts = df.select(null_exprs).collect()[0].asDict()
    total_nulls = sum(null_counts.values())

    if total_nulls == 0:
        print(f"{table_name:<28} {'[TODAS AS COLUNAS]':<30} {0:>12} ✅")
    else:
        for col_name, null_cnt in null_counts.items():
            if null_cnt > 0:
                print(f"{table_name:<28} {col_name:<30} {null_cnt:>12} ⚠️")

print("=" * 70)

# ---------------------------------------------------------------------------
# 5.2 Verificar municípios da 10ª CRS
# ---------------------------------------------------------------------------
print("\n📍 Municípios na tabela exames_geral:")
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral").select("codigo_municipio", "municipio").distinct().orderBy("codigo_municipio").show(truncate=False)